# 06 · Modelo de pronóstico a 24 h

**Objetivo:** pronosticar cloro, turbidez y pH de cada sensor 24 h adelante y evaluar contra la
meta del proyecto (`calidad_agua/metas.py`):

> al menos el **80 %** de los pronósticos a 24 h dentro de la tolerancia en cloro (± 0,1 mg/L),
> turbidez (± 20 %, mínimo ± 0,2 UNT) y pH (± 0,1), en operación normal, **y mejor que el
> pronóstico ingenuo** "igual que ayer a la misma hora" en cada parámetro.

**Enfoque** (según lo aprendido en el EDA):

- **Un modelo por parámetro, compartido por los 21 sensores.** El sensor es una variable más, junto
  con su valor típico, porque el cloro normal cambia mucho de un sensor a otro (notebook 02).
- **Se pronostica el cambio, no el valor.** El modelo aprende cuánto se va a apartar el valor de
  dentro de 24 h del valor actual (que es la línea base). Solo tiene que aprender lo que la línea
  base no explica.
- **Variables de entrada** (`calidad_agua/variables.py`): historia reciente del sensor (últimas 24 h), los
  demás parámetros del sensor, la fuente (J280) y la mediana de la red, porque los cambios de la
  fuente llegan con 1 a 15 h de retraso (notebook 03); hora del día; lluvia de las últimas 6, 24 y
  72 h, **lluvia pronosticada para las próximas 24 h** (por tramos de 0–6, 6–12 y 12–24 h) y caudal
  de la fuente. La época del año no se usa: la sección 4 muestra por qué.
- **Pronóstico de lluvia** (`calidad_agua/clima.py`): promedio de cuatro modelos meteorológicos
  globales (ECMWF, GFS, ICON y GEM), tal como estaban **24 h antes** de cada hora (Previous Runs API
  de Open-Meteo, notebook 04). Así el modelo aprende con pronósticos que se equivocan como en la
  realidad, y no usa nada que no existiera en el momento de pronosticar.
- **Algoritmo:** `HistGradientBoostingRegressor` de scikit-learn (árboles de decisión potenciados),
  con error absoluto como función de pérdida. La turbidez se modela en escala logarítmica.

**Entradas y objetivos se limpian distinto, a propósito.** Las **entradas** se limpian solo con las
reglas que existen en operación (presión, rango físico y sensor pegado:
`calidad.lecturas_operativas`), porque en vivo no hay etiquetas de falla y el modelo tiene que
aprender a convivir con lecturas con deriva, picos o ruido. Los **objetivos** son el valor real,
limpio de fallas del sensor (`calidad.lecturas_validas`), porque lo que se quiere pronosticar es el
agua, no la falla del instrumento.

**Uso de las particiones:** se entrena con entrenamiento (sep 2024 – dic 2025), se toman las
decisiones mirando validación (ene – abr 2026) y la prueba (may – ago 2026) se evalúa una sola vez al
final.

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))          # por si calidad_agua no está instalado (pip install -e .)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.inspection import permutation_importance

from calidad_agua import carga, calidad, metas, modelo, variables, clima
from calidad_agua.graficos import estilo, guardar, COLORES, GRIS

estilo()
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

crudo = carga.cargar_todo()
t15 = calidad.lecturas_operativas(crudo)        # ENTRADAS: limpieza que también existe en operación
t15_real = calidad.lecturas_validas(crudo)      # OBJETIVOS: valor real, sin fallas del sensor
h = variables.horario(t15)
h_real = variables.horario(t15_real)
pronostico = clima.ventanas(clima.cargar_pronostico(), h.index)   # lluvia pronosticada por tramos
tabla, tipicos = variables.tabla(h, horizonte=metas.HORIZONTE_H, h_objetivo=h_real, pronostico=pronostico)
X = variables.columnas_entrada(tabla)
PARAM = metas.PARAMETROS_META
ETIQ = {"CL2": "Cloro libre", "TURB": "Turbidez", "PH": "pH"}

entr = tabla["particion"] == "entrenamiento"
val = tabla["particion"] == "validacion"
prueba = tabla["particion"] == "prueba"
print(f"Tabla: {len(tabla):,} filas (sensor × hora) · {len(X)} variables de entrada")
print({p: int(m.sum()) for p, m in [("entrenamiento", entr), ("validación", val), ("prueba", prueba)]})

Tabla: 367,920 filas (sensor × hora) · 85 variables de entrada
{'entrenamiento': 245448, 'validación': 60480, 'prueba': 61992}


## 1. Entrenamiento

Un modelo por parámetro. `early_stopping` separa internamente un 10 % del entrenamiento para decidir
cuántos árboles usar, así que la validación no se toca para eso.

In [2]:
modelos, tiempos = {}, {}
for p in PARAM:
    t0 = time.time()
    modelos[p] = modelo.entrenar(tabla, p, X, entr)
    tiempos[p] = time.time() - t0
    print(f"{p}: {modelos[p].n_iter_} árboles · {tiempos[p]:.0f} s")

CL2: 400 árboles · 24 s


TURB: 400 árboles · 23 s


PH: 400 árboles · 23 s


## 2. Resultado en validación

Para cada parámetro: porcentaje de pronósticos dentro de la tolerancia (el criterio de la meta) y
error absoluto medio, del modelo y de la línea base, por separado en operación normal y durante
eventos (según si hay evento visible en el sensor a la hora pronosticada).

In [3]:
def evaluar(filas, pronosticos=None):
    resultados = []
    for p in PARAM:
        d = tabla[filas & tabla[f"objetivo_{p}"].notna() & tabla[p].notna()]
        obs = d[f"objetivo_{p}"].values
        pron = modelo.pronosticar(modelos[p], d, p, X) if pronosticos is None else pronosticos[p]
        base = d[p].values
        for cond, m in [("normal", d["evento_objetivo"].values == 0), ("evento", d["evento_objetivo"].values > 0)]:
            fila = {"parámetro": p, "condición": cond, "pronósticos": int(m.sum())}
            for nombre, x in [("modelo", pron), ("línea base", base)]:
                fila[f"% acierto {nombre}"] = 100 * np.nanmean(metas.acierto(p, obs[m], x[m]))
                fila[f"error medio {nombre}"] = np.nanmean(np.abs(obs[m] - x[m]))
            resultados.append(fila)
    r = pd.DataFrame(resultados).set_index(["parámetro", "condición"])
    r["mejora (puntos)"] = r["% acierto modelo"] - r["% acierto línea base"]
    return r[["pronósticos", "% acierto modelo", "% acierto línea base", "mejora (puntos)",
              "error medio modelo", "error medio línea base"]]

def cumple(r):
    n = r.xs("normal", level="condición")
    return pd.DataFrame({
        "% acierto (normal)": n["% acierto modelo"],
        "≥ 80 %": n["% acierto modelo"] >= metas.META_PCT,
        "mejor que la línea base": n["% acierto modelo"] > n["% acierto línea base"],
        "cumple la meta": (n["% acierto modelo"] >= metas.META_PCT) & (n["% acierto modelo"] > n["% acierto línea base"]),
    })

res_val = evaluar(val)
res_val

pronósticos  % acierto modelo  % acierto línea base  mejora (puntos)  error medio modelo  error medio línea base
parámetro condición                                                                                                                  
CL2       normal           56769            90.067                80.926            9.141               0.048                   0.076
          evento            3379            18.733                18.112            0.621               0.365                   0.381
TURB      normal           56781            71.438                62.479            8.959               0.206                   0.297
          evento            3379            52.974                51.998            0.977               0.556                   0.621
PH        normal           56742            88.543                84.248            4.295               0.049                   0.063
          evento            3379            73.542                61.409           12.134               0.198                   0.219

In [4]:
cumple(res_val)

,% acierto (normal),≥ 80 %,mejor que la línea base,cumple la meta
parámetro,,,,
CL2,90.067,True,True,True
TURB,71.438,False,True,False
PH,88.543,True,True,True


### Por sensor y por mes

¿El modelo funciona parejo en toda la red y en todo el periodo, o hay sensores o meses donde falla?

In [5]:
d = tabla[val & (tabla["evento_objetivo"] == 0)].copy()
for p in PARAM:
    ok = d[f"objetivo_{p}"].notna() & d[p].notna()
    d[f"ok_{p}"] = np.nan
    d.loc[ok, f"ok_{p}"] = metas.acierto(p, d.loc[ok, f"objetivo_{p}"], modelo.pronosticar(modelos[p], d[ok], p, X))
    d[f"base_{p}"] = metas.acierto(p, d[f"objetivo_{p}"], d[p])

por_sensor = d.groupby("nodo")[[f"ok_{p}" for p in PARAM]].mean().mul(100)
por_sensor.columns = PARAM
por_sensor = por_sensor.sort_values("CL2")
d["mes"] = d["fecha"].dt.strftime("%Y-%m")
por_mes = d.groupby("mes")[[f"ok_{p}" for p in PARAM]].mean().mul(100)
por_mes.columns = PARAM

fig, axes = plt.subplots(1, 2, figsize=(11, 6), gridspec_kw={"width_ratios": [1.3, 1]})
sns.heatmap(por_sensor, annot=True, fmt=".0f", cmap="Blues", vmin=0, vmax=100, cbar=False, ax=axes[0],
            linewidths=1, linecolor="white", annot_kws={"fontsize": 7})
axes[0].set_title("% de acierto por sensor (validación, normal)", fontsize=10); axes[0].set_ylabel("")
sns.heatmap(por_mes, annot=True, fmt=".0f", cmap="Blues", vmin=0, vmax=100, cbar=False, ax=axes[1],
            linewidths=1, linecolor="white", annot_kws={"fontsize": 8})
axes[1].set_title("% de acierto por mes", fontsize=10); axes[1].set_ylabel("")
for ax in axes:
    ax.grid(False)
fig.tight_layout()
guardar(fig, "06_acierto_sensor_mes"); plt.show()

### Un ejemplo

Pronóstico a 24 h contra lo observado en un sensor intermedio (J224) durante dos semanas de
validación. La línea punteada es la línea base: el valor de 24 h antes. Cada punto del pronóstico se hizo 24 h antes, con la información disponible en ese
momento.

In [6]:
n_ej = "J224"
ej = tabla[val & (tabla["nodo"] == n_ej)].set_index("fecha")
ini = carga.periodos()["validacion"][0] + pd.DateOffset(months=2)   # dos semanas a mitad de la validación
ventana = ej.loc[ini: ini + pd.Timedelta(days=14) - pd.Timedelta(hours=1)]
fig, axes = plt.subplots(3, 1, figsize=(11, 7.5), sharex=True)
for ax, p in zip(axes, PARAM):
    fechas_obj = ventana.index + pd.Timedelta(hours=24)
    obs = ventana[f"objetivo_{p}"].values
    pron = modelo.pronosticar(modelos[p], ventana.reset_index(), p, X)
    tol = metas.tolerancia(p, obs)
    ax.fill_between(fechas_obj, obs - tol, obs + tol, color=COLORES[0], alpha=0.12, lw=0, label="tolerancia")
    ax.plot(fechas_obj, obs, color=COLORES[0], lw=1.2, label="observado")
    ax.plot(fechas_obj, pron, color=COLORES[1], lw=1.2, label="pronóstico a 24 h")
    ax.plot(fechas_obj, ventana[p].values, color=GRIS, lw=0.8, ls=":", label="línea base (igual que ayer)")
    ax.set_ylabel(f"{ETIQ[p]}")
axes[0].legend(ncol=4, fontsize=7, loc="upper left")
d1, d2 = ventana.index[0] + pd.Timedelta(hours=24), ventana.index[-1] + pd.Timedelta(hours=24)
axes[0].set_title(f"Pronóstico a 24 h en {n_ej}: fechas pronosticadas del {d1.day} {carga.MESES[d1.month - 1]} "
                  f"al {d2.day} {carga.MESES[d2.month - 1]} {d2.year} (validación)")
fig.tight_layout()
guardar(fig, "06_ejemplo_pronostico"); plt.show()

## 3. ¿Qué variables usa el modelo?

Importancia por permutación en una muestra de validación: cuánto empeora el error si se desordena
cada variable. Se muestran las 12 más importantes de cada modelo.

In [7]:
muestra = tabla[val].sample(15000, random_state=0)
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
importancias = {}
for ax, p in zip(axes, PARAM):
    m = muestra[f"objetivo_{p}"].notna() & muestra[p].notna()
    r = permutation_importance(modelos[p], muestra.loc[m, X], modelo.objetivo(muestra[m], p),
                               n_repeats=3, random_state=0, scoring="neg_mean_absolute_error")
    imp = pd.Series(r.importances_mean, index=X).sort_values(ascending=False).head(12)
    importancias[p] = imp
    ax.barh(imp.index[::-1], imp.values[::-1], color=COLORES[0])
    ax.set_title(ETIQ[p], fontsize=10)
    ax.tick_params(axis="y", labelsize=7)
fig.suptitle("Variables más importantes (aumento del error al desordenarlas)", x=0.01, ha="left",
             fontsize=11, fontweight="bold")
fig.tight_layout()
guardar(fig, "06_importancia_variables"); plt.show()

## 4. ¿Qué información de clima conviene usar?

La turbidez y el pH dependen de la lluvia, y en la época lluviosa son los parámetros más difíciles
(notebook 03). Se comparan cuatro versiones del modelo que solo cambian en las variables de clima:

| Versión | Época del año | Pronóstico de lluvia |
|---|---|---|
| con época del año | sí (`anio_sin`, `anio_cos`) | no |
| sin época del año | no | no |
| **sin época + pronóstico** (el modelo de este notebook) | no | promedio de 4 modelos, hecho 24 h antes |
| techo: lluvia real | no | la lluvia que de verdad cayó (pronóstico perfecto) |

El techo **no** es un modelo utilizable: en operación esa información no existe. Dice cuánto se
ganaría con un pronóstico perfecto.

**Dos validaciones.** La validación oficial (ene–abr 2026) es la época más seca y casi no pone a
prueba la lluvia. Por eso se agrega una segunda: se entrena con sep 2024 – ago 2025 y se evalúa en
**sep–dic 2025**, un bloque lluvioso dentro del periodo de entrenamiento. La prueba no se usa para
decidir.

En pruebas previas también se probó agregar la lluvia de 12 h, 48 h y 30 días: no mejoró en ninguna
de las dos validaciones, así que no se incluye. Y el pronóstico "best_match" de Open-Meteo, en vez
del promedio de cuatro modelos, empeoraba al modelo (notebook 04).

In [8]:
lluvia = h["lluvia_mm"]
def proximas(a, b):          # lluvia real de las horas t+a+1 … t+b (lo que el pronóstico intenta anticipar)
    suma = lambda k: lluvia[::-1].rolling(k, min_periods=1).sum()[::-1].shift(-1) if k else 0
    return suma(b) - suma(a)
extra = pd.DataFrame({f"lluvia_real_{a}_{b}h": proximas(a, b) for a, b in clima.TRAMOS})
dia = extra.index.dayofyear
extra["anio_sin"], extra["anio_cos"] = np.sin(2 * np.pi * dia / 365.25), np.cos(2 * np.pi * dia / 365.25)
tabla_exp = tabla.merge(extra, left_on="fecha", right_index=True, how="left")

sin_pron = [c for c in X if c not in clima.COLUMNAS]
VERSIONES = {
    "con época del año": sin_pron + ["anio_sin", "anio_cos"],
    "sin época del año": sin_pron,
    "sin época + pronóstico de lluvia": X,
    "techo: lluvia real": sin_pron + [f"lluvia_real_{a}_{b}h" for a, b in clima.TRAMOS],
}
f = tabla_exp["fecha"]
VALIDACIONES = {
    f"validación ({carga.texto_periodo('validacion')})": (entr, val),
    "bloque lluvioso (sep–dic 2025)": (entr & (f < "2025-09-01"), (f >= "2025-09-01") & (f < "2026-01-01")),
}
filas = []
for nombre_v, (f_entr, f_eval) in VALIDACIONES.items():
    for p in PARAM:
        d = tabla_exp[f_eval & tabla_exp[f"objetivo_{p}"].notna() & tabla_exp[p].notna() & (tabla_exp["evento_objetivo"] == 0)]
        obs = d[f"objetivo_{p}"].values
        filas.append({"validación": nombre_v, "parámetro": p, "versión": "línea base",
                      "% acierto": 100 * np.nanmean(metas.acierto(p, obs, d[p].values))})
        for nombre, cols in VERSIONES.items():
            m = modelo.entrenar(tabla_exp, p, cols, f_entr)
            filas.append({"validación": nombre_v, "parámetro": p, "versión": nombre,
                          "% acierto": 100 * np.nanmean(metas.acierto(p, obs, modelo.pronosticar(m, d, p, cols)))})
clima_exp = pd.DataFrame(filas).pivot_table(index="versión", columns=["validación", "parámetro"], values="% acierto", sort=False)
clima_exp.round(1)

validación                       validación (ene – abr 2026)             bloque lluvioso (sep–dic 2025)            
parámetro                                                CL2  TURB    PH                            CL2  TURB    PH
versión                                                                                                            
línea base                                              80.9  62.5  84.2                           82.4  47.5  72.2
con época del año                                       88.3  66.3  85.9                           89.2  29.9  68.2
sin época del año                                       89.3  65.7  86.7                           90.2  48.7  81.0
sin época + pronóstico de lluvia                        90.1  71.4  88.5                           90.0  53.5  82.8
techo: lluvia real                                      91.3  90.4  93.9                           90.5  85.6  94.0

## 5. Evaluación final en el periodo de prueba

Se evalúa una sola vez, con los modelos ya fijados. Es el resultado que se reporta.

In [9]:
res_prueba = evaluar(prueba)
res_prueba

pronósticos  % acierto modelo  % acierto línea base  mejora (puntos)  error medio modelo  error medio línea base
parámetro condición                                                                                                                  
CL2       normal           56684            87.815                78.886            8.928               0.052                   0.073
          evento            4186            51.696                49.498            2.198               0.195                   0.200
TURB      normal           56853            48.727                42.246            6.482               0.346                   0.582
          evento            4218            24.656                46.325          -21.669               2.763                   2.925
PH        normal           56877            83.832                77.578            6.254               0.058                   0.077
          evento            4234            60.156                68.068           -7.912               0.233                   0.216

In [10]:
final = cumple(res_prueba)
final

,% acierto (normal),≥ 80 %,mejor que la línea base,cumple la meta
parámetro,,,,
CL2,87.815,True,True,True
TURB,48.727,False,True,False
PH,83.832,True,True,True


In [11]:
fig, ax = plt.subplots(figsize=(8, 3.4))
n = res_prueba.xs("normal", level="condición")
e = res_prueba.xs("evento", level="condición")
x = np.arange(len(PARAM))
ancho = 0.2
for k, (nombre, serie, color) in enumerate([("línea base, normal", n["% acierto línea base"], GRIS),
                                            ("modelo, normal", n["% acierto modelo"], COLORES[0]),
                                            ("línea base, eventos", e["% acierto línea base"], "#c3c2b7"),
                                            ("modelo, eventos", e["% acierto modelo"], COLORES[1])]):
    barras = ax.bar(x + (k - 1.5) * ancho, serie.values, ancho * 0.92, color=color, label=nombre)
    for b_, v_ in zip(barras, serie.values):
        ax.text(b_.get_x() + b_.get_width() / 2, v_ + 1, f"{v_:.0f}", ha="center", fontsize=7, color="#52514e")
ax.axhline(metas.META_PCT, color=COLORES[7], ls="--", lw=1)
ax.text(len(PARAM) - 0.5, metas.META_PCT + 1, "meta 80 %", color=COLORES[7], fontsize=8, ha="right")
ax.set_xticks(x); ax.set_xticklabels([f"{ETIQ[p]}\n({metas.texto_tolerancia(p)})" for p in PARAM])
ax.set_ylim(0, 105); ax.set_ylabel("% dentro de la tolerancia")
ax.set_title(f"Pronóstico a 24 h en el periodo de prueba ({carga.texto_periodo('prueba')})")
ax.legend(ncol=2, fontsize=7, loc="lower left")
guardar(fig, "06_resultado_prueba"); plt.show()

## 6. Guardar los modelos

Se guardan en `modelos/` junto con lo necesario para usarlos después (por ejemplo, desde el panel):
las columnas de entrada y el valor típico de cada sensor.

In [12]:
carpeta = Path.cwd().parent / "modelos"
carpeta.mkdir(exist_ok=True)
joblib.dump({"modelos": modelos, "columnas": X, "tipicos": tipicos, "horizonte_h": metas.HORIZONTE_H,
             "sensores": sorted(tabla["nodo"].unique(), key=list(tabla["nodo"].unique()).index)},
            carpeta / "pronostico_24h.joblib", compress=3)
print("Guardado:", carpeta / "pronostico_24h.joblib",
      f"({(carpeta / 'pronostico_24h.joblib').stat().st_size / 1e6:.1f} MB)")

Guardado: /home/claude/work/Modelo_calidad_agua/modelos/pronostico_24h.joblib (2.2 MB)


## 7. Conclusiones

**Resultado en el periodo de prueba (may–ago 2026, época lluviosa), operación normal**

| Parámetro | Acierto del modelo | Línea base | Antes (con época, sin pronóstico) | ¿Cumple la meta? |
|---|---|---|---|---|
| Cloro libre (± 0,1 mg/L) | **88 %** | 79 % | 86 % | **Sí** |
| Turbidez (± 20 %, mín. 0,2 UNT) | 49 % | 42 % | 37 % | No: mejora la línea base, pero no llega al 80 % |
| pH (± 0,1) | **84 %** | 78 % | 74 % | **Sí** |

En validación (ene–abr 2026, la época más seca): cloro 90 % (línea base 81 %), turbidez 71 %
(62 %) y pH 89 % (84 %).

**Lo que muestra el modelo**

1. **Quitar la época del año fue la mejora más grande.** En el bloque lluvioso (sep–dic 2025), sin
   las variables de época la turbidez pasa de 30 % a 49 % y el pH de 68 % a 81 %; en la validación
   oficial casi no cambia. Con 16 meses de entrenamiento el modelo ve cada época una sola vez y
   aprende cómo fue *ese* año, que no se repite: junio de 2025 tuvo 745 mm de lluvia y junio de
   2026, 190 mm. Sin la época, el modelo tiene que apoyarse en la lluvia, que sí explica lo que pasa.
2. **El pronóstico de lluvia ayuda en las dos validaciones.** Sobre el modelo sin época, la turbidez
   sube de 66 % a 71 % en validación y de 49 % a 54 % en el bloque lluvioso, y el pH 2 puntos en
   las dos. Ahora son de las variables que más usa el modelo de turbidez, después del valor
   actual. Solo funciona con el promedio de cuatro modelos meteorológicos: con "best_match" el
   modelo empeoraba (notebook 04).
3. **El cloro cumple en las dos épocas.** Sube de 79 % a 88 % en prueba y baja el error medio de
   0,073 a 0,052 mg/L. Usa sobre todo el valor actual, el sensor, la mediana de la red y el valor de
   hace 24 h: nada de eso depende del clima.
4. **El pH ahora cumple también en la época lluviosa** (84 % en prueba, contra 78 % de la línea
   base). Antes quedaba por debajo de la línea base.
5. **La turbidez mejoró mucho, pero no llega al 80 % en la época lluviosa.** De 37 % a 49 % en
   prueba, 7 puntos sobre la línea base. El techo con lluvia perfecta es 86–90 %: lo que falta no
   es el modelo sino la calidad del pronóstico de lluvia, que acierta a medias en una zona de
   aguaceros de tarde (correlación 0,65, notebook 04).
6. **Durante eventos el modelo casi no mejora la línea base.** En prueba, cloro 52 % contra 49 %, y
   en turbidez y pH queda por debajo. Un pronóstico a 24 h no puede anticipar un evento que empieza
   de repente; eso le toca al detector (notebook 07).
7. **Hay sensores y meses más difíciles.** En cloro, J123, J345 y J69 quedan entre 71 % y 79 % en
   validación, contra más de 85 % en la mayoría. Febrero de 2026 es el peor mes (turbidez 54 %, pH
   77 %): fue un febrero muy lluvioso (411 mm) y con un frente frío.

**Siguientes pasos**

- **Notebook 08:** comparar con la red LSTM y elegir el modelo final por parámetro.
- **En operación:** el servicio necesita el pronóstico vigente de los cuatro modelos (Forecast API
  de Open-Meteo) para armar las mismas variables.
